In [ ]:
%load_ext autoreload
%autoreload 2

# Interaction recovery quality analysis (interaction information)

Edge selection is based **exclusively** on raw interaction information,
`ii = II(i; j; Y)`, computed for each feature pair. The pair score is treated as
a score in a classification task: true vs spurious pair. The ground truth comes
from the `pairwise` generator with the default interaction set (`DEFAULT_INTERACTIONS`).

Quality metrics are computed directly from the `ii` scores (without building graphs):
- **ROC AUC** and **PR AUC** - how well `ii` ranks true pairs above spurious ones,
- **precision / recall / F1** for top-`TOP_K` predictions,

as functions of training size, feature correlation (`cov`), and the **number of
discretization bins (`n_bins`)**, which is the main dimension of all plots.


In [ ]:
import dotenv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from my_project.data.generators import (
    DEFAULT_INTERACTIONS,
    build_generator,
    compute_config_hash,
    interactions_name_map,
    pretty_interaction_name,
)
from my_project.data.utils import prepare_data
from my_project.information_theory import compute_info
from my_project.reporting import (
    CATEGORICAL_PALETTE,
    SEQUENTIAL_PALETTE,
    configure_notebook_plots,
    pair_table_from_matrix,
    plot_ii_graph,
    save_figure,
    save_table,
)

dotenv.load_dotenv(override=True)

configure_notebook_plots()
PALETTE = "husl"
PALETTE_SEQ = SEQUENTIAL_PALETTE
PALETTE_CAT = CATEGORICAL_PALETTE
GRAY = (0.65, 0.65, 0.65)

## Configuration

Generator selection uses one cell per generator. To switch generators, **comment out
the entire cell** for the generator you do not want, leaving exactly one uncommented
(if both are active, the lower one takes precedence). Shared settings (grid, criterion,
`n_bins`) are in a separate cell, and the final cell computes all derived values
(`generate`, `GENERATOR_HASH`, true edges, `TOP_K`) from the **final** configuration.
Thus, switching generators does not leave stale values. True edges come directly from
the generator (`true_interactions`): for `pairwise`, these are the declared pairs; for
`higgs`, they form a physics-based graph.


In [ ]:
# ======================================================================
# Generator: PAIRWISE (comment out the ENTIRE cell to select another generator)
# ======================================================================

GENERATOR_KIND = "pairwise"
N_INFORMATIVE = 10

# True interaction pairs. By default, use the set from the module (identical to the
# one used in modeling); a custom set can be specified below.
INTERACTIONS = DEFAULT_INTERACTIONS
# INTERACTIONS = {
#     ("x1", "x2"): sign_times_another,
#     ("x3", "x4"): multiplication,
#     ("x5", "x6"): sinusoidal_interaction,
#     ("x7", "x8"): xor,
#     ("x6", "x8"): quadratic_magnitude_interaction,
#     ("x9", "x10"): similarity_interaction,
# }

GENERATOR_KWARGS = {
    "n_informative": N_INFORMATIVE,
    "coef_scale": 0.0,
    "coef_loc": 1.0,
    "interactions": INTERACTIONS,
}

In [ ]:
# # ======================================================================
# # Generator: HIGGS (comment out the ENTIRE cell to select another generator)
# # ======================================================================
# # The true HIGGS interaction graph comes from physics (invariant masses),
# # not from INTERACTIONS; the generator returns it in `true_interactions`.
# GENERATOR_KIND = "higgs"
# GENERATOR_KWARGS = {}

In [ ]:
# Shared settings (independent of the selected generator).
N_BINS = 5

# Split sizes IDENTICAL to those in the `synthetic_data` notebook (TEST_SIZE=5000,
# VALID_SIZE=5000). The training split comes from a stratified train_test_split that
# depends on the test/validation sizes, so using the same values guarantees that for
# a given (train_size, cov, noise, data_seed), EXACTLY the same rows are used for
# training as in `synthetic_data`.
TEST_SIZE = 5000
VALID_SIZE = 5000

# Full grid (uncomment for the actual run):
TRAIN_SIZE_GRID = [1000, 2500, 5000, 10000, 15000, 20000, 25000]
COV_GRID = [0.0, 0.25, 0.5]
N_BINS_GRID = [3, 5, 10]
DATA_SEEDS = list(range(20))

NOISE_GRID = [0.0]

# Quick test run:
# TRAIN_SIZE_GRID = [1000, 2000, 3000]
# COV_GRID = [0.0]
# NOISE_GRID = [0.0]
# N_BINS_GRID = [5]
# DATA_SEEDS = list(range(10))

In [ ]:
# Build the generator and compute values derived from the FINAL configuration, so
# switching generators (by commenting out one of the cells above) automatically
# updates the hash, true edges, and TOP_K.
generate = build_generator(GENERATOR_KIND, GENERATOR_KWARGS)

# True edges directly from the generator (`true_interactions`): declared pairs for
# pairwise or a physics-based graph for HIGGS. This is the same source used by the
# run loop, so N_TRUE_EDGES / TOP_K always remain consistent with the labels.
TRUE_EDGES = generate(n_samples=1000, random_state=0)["true_interactions"]
N_TRUE_EDGES = len(TRUE_EDGES)
# Prediction = TOP_K pairs with the highest score (instead of a value threshold).
TOP_K = N_TRUE_EDGES

# Interaction names only for generators that define them (e.g. pairwise); real-world
# datasets (HIGGS) do not have them.
INTERACTIONS_NAMES = (
    interactions_name_map(GENERATOR_KWARGS["interactions"])
    if "interactions" in GENERATOR_KWARGS
    else {}
)

# Consistent interaction-type palette: each type receives a fixed PALETTE_CAT color
# (shared by all plots), while spurious/non-interacting pairs are gray.
# Type labels use mathematical notation (PRETTY_NAMES from the generator).
NONINT_LABEL = "pozostałe"

_typ_names = list(
    dict.fromkeys(
        pretty_interaction_name(func)
        for func in GENERATOR_KWARGS.get("interactions", {}).values()
    )
)
TYP_PALETTE = dict(
    zip(_typ_names, sns.color_palette(PALETTE_CAT, max(len(_typ_names), 1)))
)
TYP_PALETTE[NONINT_LABEL] = GRAY

GENERATOR_HASH = compute_config_hash(GENERATOR_KIND, GENERATOR_KWARGS, 0.0, N_BINS)

print("generator:", GENERATOR_KIND)

print("interactions:", INTERACTIONS_NAMES)
print(
    "n_true_edges:", N_TRUE_EDGES, "| top_k:", TOP_K, "| config_hash:", GENERATOR_HASH
)

## Runs: score computation (expensive)

For each `(noise, cov, train_size, n_bins, data_seed)` combination, we generate data,
compute the `ii` matrix on the training set, and store pair `scores` (upper triangle)
and `labels` (true/spurious). This is the only expensive step: data is generated once
per `(noise, cov, train_size, seed)`, while `n_bins` changes only when computing `ii`.
Metrics are computed separately, so changing `TOP_K` does not require rerunning this
cell.

In [ ]:
def pair_scores(ii, true_edges):
    """Return (scores, labels) for all feature pairs in the upper triangle of ii."""
    mat = ii.to_numpy()
    cols = list(ii.columns)
    iu, ju = np.triu_indices(len(cols), k=1)
    scores = mat[iu, ju]
    true_set = {frozenset(e) for e in true_edges}
    labels = np.array(
        [frozenset((cols[i], cols[j])) in true_set for i, j in zip(iu, ju)]
    )
    return scores, labels

In [ ]:
scored = []
for noise_level in NOISE_GRID:
    for cov in COV_GRID:
        for train_size in TRAIN_SIZE_GRID:

            print(f"noise={noise_level:.2f}, cov={cov:.2f}, train_size={train_size}")

            for data_seed in DATA_SEEDS:
                n_samples = train_size + TEST_SIZE + VALID_SIZE
                true_edges, splits = prepare_data(
                    generate,
                    n_samples,
                    data_seed,
                    test_size=TEST_SIZE,
                    valid_size=VALID_SIZE,
                    noise_level=noise_level,
                    cov=cov,
                )
                X_train, y_train = splits[0], splits[3]

                for n_bins in N_BINS_GRID:
                    # Edge selection = raw ii = II(i; j; Y) for each feature pair
                    # (upper matrix triangle); the label indicates whether the pair
                    # belongs to the true interaction set.
                    ii = compute_info(X_train, y_train, n_bins=n_bins)
                    ii_scores, labels = pair_scores(ii, true_edges)

                    scored.append(
                        {
                            "noise_level": noise_level,
                            "cov": cov,
                            "train_size": train_size,
                            "n_bins": n_bins,
                            "data_seed": data_seed,
                            "ii": ii_scores,
                            "labels": labels,
                        }
                    )

print("zapisano score'ów:", len(scored))

## Scoring (inexpensive) - change `TOP_K` and rerun from this cell

Metrics are computed from the stored `ii` scores. `roc_auc`/`pr_auc` are threshold-
independent; `precision`/`recall`/`f1` are computed for top-`TOP_K` predictions.


In [ ]:
def top_k_pred(scores, k):
    """Prediction mask: mark the k pairs with the highest scores as positive."""
    pred = np.zeros(len(scores), dtype=bool)
    pred[np.argsort(-scores)[:k]] = True
    return pred


def score_metrics(scores, labels, k):
    """ROC/PR AUC (threshold-independent) and precision/recall/F1 for top-k."""
    pred = top_k_pred(scores, k)
    return {
        "roc_auc": roc_auc_score(labels, scores),
        "pr_auc": average_precision_score(labels, scores),
        "precision": precision_score(labels, pred, zero_division=0),
        "recall": recall_score(labels, pred, zero_division=0),
        "f1": f1_score(labels, pred, zero_division=0),
    }


# Metrics computed from raw ii for each grid cell (in long format).
records = []
for rec in scored:
    records.append(
        {
            "noise_level": rec["noise_level"],
            "cov": rec["cov"],
            "train_size": rec["train_size"],
            "n_bins": rec["n_bins"],
            "data_seed": rec["data_seed"],
            **score_metrics(rec["ii"], rec["labels"], TOP_K),
        }
    )

results_df = pd.DataFrame(records)
results_df.head()

## Aggregation across seeds

Mean metrics across `data_seed` values for each grid cell.

In [ ]:
METRICS = ["roc_auc", "pr_auc", "precision", "recall", "f1"]
agg = results_df.groupby(
    ["noise_level", "cov", "train_size", "n_bins"], as_index=False
)[METRICS].mean()
agg

## Curves: quality vs training size by number of bins

Plot grid: rows = `cov`, columns = `noise_level`, **lines = `n_bins`**.
The X-axis shows `train_size`. ROC AUC and PR AUC use separate grids.


In [ ]:
plot_df = results_df.copy()
plot_df["train_size"] = pd.Categorical(
    plot_df["train_size"], categories=TRAIN_SIZE_GRID, ordered=True
)
plot_df["n_bins"] = plot_df["n_bins"].astype("category")
plot_df["liczba przedziałów"] = plot_df["n_bins"].copy()
hue_var = "liczba przedziałów"

titles = {
    "roc_auc": "roc auc",
    "pr_auc": "pr auc",
}

if len(plot_df["noise_level"].drop_duplicates()) == 1:
    row_col = None
else:
    row_col = "noise_level"

for metric in ["roc_auc", "pr_auc"]:
    g = sns.relplot(
        data=plot_df,
        kind="line",
        x="train_size",
        y=metric,
        hue=hue_var,
        style=hue_var,
        col="cov",
        row=row_col,
        marker="o",
        palette=PALETTE_CAT,
        height=2.6,
        aspect=1.3,
        errorbar="sd",
        # err_style="bars",
        # err_kws={"capsize": 3},
        facet_kws={"margin_titles": True},
    )
    # g.set(ylim=(0, 1.05))
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=45)

    g.set_titles(row_template="{row_name}", col_template="{col_name}")
    middle_col = (g.axes.shape[1] - 1) // 2
    middle_col_title = g.axes[0, middle_col].get_title()
    g.axes[0, middle_col].set_title(f"współczynnik korelacji\n{middle_col_title}")
    g.set_xlabels("")
    g.axes[-1, middle_col].set_xlabel("liczność zbioru treningowego")
    # g.figure.suptitle(titles[metric], y=1.02)

    save_figure(g.figure, f"recovery_ii_{metric}_vs_train_size")

    plt.show()

    plot_df_agg = plot_df.groupby(
        ["noise_level", "cov", "train_size", hue_var], as_index=False, observed=False
    )[metric].agg(["mean", "std"])
    save_table(plot_df_agg.round(3), f"recovery_ii_{metric}_vs_train_size")

## Separation of true and spurious pairs

For one "easy" dataset (large training set, no noise or correlation), we sort all
feature pairs by their `ii` score. Good separation means that true pairs are
concentrated at the top of the ranking, within the `TOP_K` boundary.

In [ ]:
n_samples = 25000 + TEST_SIZE + VALID_SIZE
true_edges, splits = prepare_data(
    generate=generate,
    n_samples=n_samples,
    data_seed=42,
    test_size=TEST_SIZE,
    valid_size=VALID_SIZE,
    noise_level=0.0,
    cov=0.0,
)
X_train, y_train = splits[0], splits[3]
ii = compute_info(X_train, y_train, n_bins=N_BINS)

scores, labels = pair_scores(ii, true_edges)
order = np.argsort(-scores)
pairs = pd.DataFrame(
    {
        "rank": np.arange(1, len(scores) + 1),
        "interaction_information": scores[order],
        "is_true": labels[order],
    }
)

# Consistent with the other plots: true pairs use PALETTE_CAT, spurious pairs are gray.
sns.scatterplot(
    data=pairs,
    x="rank",
    y="interaction_information",
    hue="is_true",
    palette={True: sns.color_palette(PALETTE_CAT)[0], False: GRAY},
)
plt.axvline(TOP_K + 0.5, color="red", linestyle="--", label=f"top-{TOP_K}")
plt.title("Score ii wg rangi (pary prawdziwe vs pozorne)")
plt.legend()
plt.show()

pairs[pairs["is_true"]]

## Recovered interaction graph

Graph of the `top-TOP_K` feature pairs with the highest interaction information
`II(Xi; Xj; y)`, computed on the same "easy" dataset (large training set, no noise
or correlation). This is a visual version of the scatterplot ranking: edge width
and color encode II strength.


In [ ]:
# Graph of the top-TOP_K pairs by II (function from my_project.reporting).
pair_table = pair_table_from_matrix(ii)

fig, ax = plt.subplots(figsize=(9, 9))
graph = plot_ii_graph(
    pair_table,
    ii.columns,
    k=TOP_K,
    ax=ax,
    node_size=300,
    layout="circular",
    curved_edges=False,
)
ax.set_title(f"Graf interakcji — top-{TOP_K} par wg II")
fig.tight_layout()
save_figure(fig, "recovery_ii_interaction_graph")
plt.show()

## Mean rank per interaction

For each true pair, we compute its position in the `ii` score ranking (1 = highest
score) in every run and then average it across the entire grid. A lower mean rank
indicates an interaction that is easier to recover. Ideally, every true pair should
be within `top-N_TRUE_EDGES`.

In [ ]:
# Reference: a fixed pair order (upper triangle) shared by all runs (the same columns),
# together with the indices and types of true pairs.
_true_ref, _splits_ref = prepare_data(
    generate,
    TRAIN_SIZE_GRID[0] + TEST_SIZE + VALID_SIZE,
    0,
    test_size=TEST_SIZE,
    valid_size=VALID_SIZE,
    noise_level=0.0,
    cov=0.0,
)
_ii_ref = compute_info(_splits_ref[0], _splits_ref[3], n_bins=N_BINS)
_cols = list(_ii_ref.columns)
_iu, _ju = np.triu_indices(len(_cols), k=1)
_pairs = [(_cols[i], _cols[j]) for i, j in zip(_iu, _ju)]
_type_map = {
    frozenset(pair): pretty_interaction_name(func)
    for pair, func in INTERACTIONS.items()
}
_true_set = set(_type_map)
_true_idx = [k for k, (a, b) in enumerate(_pairs) if frozenset((a, b)) in _true_set]

# Rank each true pair in every run (1 = highest ii score).
rank_records = []
for rec in scored:
    order = np.argsort(-rec["ii"])
    ranks = np.empty(len(order), dtype=int)
    ranks[order] = np.arange(1, len(order) + 1)
    for k in _true_idx:
        a, b = _pairs[k]
        typ = _type_map[frozenset((a, b))]
        rank_records.append(
            {
                "noise_level": rec["noise_level"],
                "cov": rec["cov"],
                "train_size": rec["train_size"],
                "n_bins": rec["n_bins"],
                "interaction": f"{a}×{b}",
                "typ": typ,
                "label": f"{a} × {b}: {typ}",
                "rank": ranks[k],
                "ii": rec["ii"][k],
            }
        )

rank_df = pd.DataFrame(rank_records)

In [ ]:
mean_rank = (
    rank_df.groupby(["label", "interaction", "typ"])["rank"]
    .agg(["mean", "std"])
    .sort_values("mean")
    .reset_index()
)

In [ ]:
plt.figure(figsize=(8, 8))
sns.boxplot(
    data=rank_df,
    y="rank",
    x="label",
    hue="typ",
    order=mean_rank["label"],
    palette=TYP_PALETTE,
    # errorbar="sd",
    dodge=False,
    legend=False,
)
plt.axhline(
    N_TRUE_EDGES + 0.5, color="red", linestyle="--", label=f"True edges-{N_TRUE_EDGES}"
)
plt.ylabel("rank")
plt.xlabel("interakcja")
plt.title("Średni ranking prawdziwych par interakcji")
plt.xticks(rotation=45, ha="right")
# plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Mean `ii` value per interaction

The same analysis, but using raw interaction information values instead of ranks.
A higher `ii` indicates a stronger interaction signal. This complements the ranking
by showing whether differences between pairs are substantial or merely cosmetic.

In [ ]:
# Restrict grid columns to selected training sizes and a single bin count
# (mixing n_bins distorts the ii scale and obscures separation).
TRAIN_SIZES_II_PLOT = [1000, 5000, 10000]
N_BINS_II_PLOT = 5
FONT_SIZE = 20
rank_df_ii = rank_df[
    rank_df["train_size"].isin(TRAIN_SIZES_II_PLOT)
    & (rank_df["n_bins"] == N_BINS_II_PLOT)
]

# Reference bar: all NON-interacting (spurious) pairs collected into one group.
# Take ii values directly from `scored` (mask ~labels), using the same training sizes
# and bin count as for the true pairs.
nonint_records = []
for rec in scored:
    if rec["train_size"] not in TRAIN_SIZES_II_PLOT or rec["n_bins"] != N_BINS_II_PLOT:
        continue
    for ii_val in rec["ii"][~rec["labels"]]:
        nonint_records.append(
            {
                "noise_level": rec["noise_level"],
                "cov": rec["cov"],
                "train_size": rec["train_size"],
                "n_bins": rec["n_bins"],
                "interaction": NONINT_LABEL,
                "typ": NONINT_LABEL,
                "label": NONINT_LABEL,
                "ii": ii_val,
            }
        )
nonint_df = pd.DataFrame(nonint_records)

# True pairs and the non-interacting-pairs bar on shared axes.
plot_ii_df = pd.concat([rank_df_ii, nonint_df], ignore_index=True)
label_order = list(mean_rank["label"]) + [NONINT_LABEL]

mean_ii = (
    plot_ii_df.groupby(["cov", "train_size", "label", "interaction", "typ"])["ii"]
    .agg(["mean", "std"])
    .sort_values(["cov", "train_size", "mean"], ascending=[True, True, False])
    .reset_index()
)

col_var = "train_size"
row_var = "cov"
col_order = TRAIN_SIZES_II_PLOT
row_order = COV_GRID

g = sns.catplot(
    data=plot_ii_df,
    kind="box",
    y="ii",
    x="label",
    hue="typ",
    order=label_order,
    # hue_order=label_order,
    col=col_var,
    row=row_var,
    dodge=False,
    legend=False,
    row_order=row_order,
    col_order=col_order,
    # sharex=True,
    # sharex=False,
    sharey="row",
    # sharey = False,
    height=5,
    aspect=1.6,
    margin_titles=True,
    palette=PALETTE,
)

g.map_dataframe(
    sns.pointplot,
    x="label",
    y="ii",
    hue="typ",
    order=label_order,
    hue_order=label_order,
    # hue_order=label_order,
    estimator=np.mean,
    errorbar=None,
    dodge=0,
    markers="x",
    linestyles="none",
    markersize=5,
    markeredgewidth=0.6,
    # palette=PALETTE,
    # palette={label: "black" for label in label_order},
    # palette = {label: "black" for label in label_order},
    palette="dark:black",
    legend=False,
    zorder=10,
)

g.set_axis_labels("", "")

n_rows, n_cols = g.axes.shape
middle_col = (n_cols - 1) // 2
middle_row = (n_rows - 1) // 2

# g.axes[-1, middle_col].set_xlabel("pair")
g.axes[middle_row, 0].set_ylabel("informacja interakcyjna", fontsize=FONT_SIZE)

g.set_titles(row_template="{row_name}", col_template="{col_name}", size=FONT_SIZE)
middle_col_title = g.axes[0, middle_col].get_title()
g.axes[0, middle_col].set_title(
    f"wielkość zbioru treningowego\n{middle_col_title}", fontsize=FONT_SIZE
)
middle_row_title = g._margin_titles_texts[middle_row]
middle_row_title.set_text(f"współczynnik korelacji\n{middle_row_title.get_text()}")
middle_row_title.set_multialignment("center")
for margin_title in g._margin_titles_texts:
    margin_title.set_fontsize(FONT_SIZE)

# Symmetric log scale for ii with a small linear-region threshold (ii is on the
# order of 0.0-0.1) and regular, non-scientific tick labels.
LINTHRESH = 0.01
# _ii_fmt = FuncFormatter(lambda v, _: f"{v:g}")
for ax in g.axes.flat:
    # ax.axhline(0.0, color="red", linestyle="--", linewidth=1)
    # ax.set_yscale("symlog", linthresh=LINTHRESH)
    # ax.yaxis.set_major_formatter(_ii_fmt)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45)
        lbl.set_ha("right")
# g.figure.suptitle(f"Interaction information by cov and training size (n_bins={N_BINS_II_PLOT})", y=1.01)
g.tight_layout()
save_figure(g.figure, "recovery_ii_boxplot_by_cov_train")
plt.show()

mean_ii.drop(columns=["label"])

## Score vs ranking grid by correlation and training size

The grid follows the per-interaction `ii` plot: rows = `cov`, columns = `train_size`
(the same `TRAIN_SIZES_II_PLOT`). Each panel shows `ii` as a function of rank (pairs
sorted by decreasing score), with series = `n_bins` and bands = variation across
seeds. The red line marks the `top-N_TRUE_EDGES` boundary.


In [ ]:
# Grid matching the ii-per-interaction plot: rows = cov, columns = train_size
# (the same sizes as TRAIN_SIZES_II_PLOT), series = n_bins.
FONT_SIZE = 15

sr_records = []
for rec in scored:
    if rec["train_size"] not in TRAIN_SIZES_II_PLOT:
        continue
    ii_scores = rec["ii"]
    order = np.argsort(-ii_scores)
    for r, k in enumerate(order, start=1):
        sr_records.append(
            {
                "noise_level": rec["noise_level"],
                "cov": rec["cov"],
                "train_size": rec["train_size"],
                "n_bins": rec["n_bins"],
                "data_seed": rec["data_seed"],
                "rank": r,
                "ii": ii_scores[k],
            }
        )

sr_df = pd.DataFrame(sr_records)
sr_df2 = sr_df.rename(columns={"n_bins": "liczba przedziałów"})

g = sns.relplot(
    data=sr_df2,
    kind="line",
    x="rank",
    y="ii",
    # y = "rank",
    # x = "ii",
    hue="liczba przedziałów",
    row="cov",
    col="train_size",
    col_order=TRAIN_SIZES_II_PLOT,
    palette=PALETTE_SEQ,
    errorbar="sd",
    # marker="o",
    height=2.6,
    aspect=1.3,
    facet_kws={"margin_titles": True, "sharey": "row"},
    # facet_kws={"margin_titles": True, "sharey": True},
    # facet_kws={"margin_titles": True, "sharey": False},
)

n_rows, n_cols = g.axes.shape
middle_col = (n_cols - 1) // 2
middle_row = (n_rows - 1) // 2

g.figure.subplots_adjust(right=0.82)
sns.move_legend(
    g,
    "center left",
    bbox_to_anchor=(0.9, 0.5),
    ncol=1,
    frameon=False,
)

g.set_axis_labels("", "")
g.axes[-1, middle_col].set_xlabel("rank", fontsize=FONT_SIZE)
g.axes[middle_row, 0].set_ylabel("informacja interakcyjna", fontsize=FONT_SIZE)

# Symmetric log scale for ii with a small linear-region threshold (ii is on the
# order of 0.0-0.1), more ticks (subs=[1,2,3,5] adds ticks within decades), and
# regular, non-scientific labels.
LINTHRESH = 0.01
# _ii_fmt = FuncFormatter(lambda v, _: f"{v:g}")
for ax in g.axes.flat:
    ax.axvline(N_TRUE_EDGES + 0.5, color="red", linestyle="--", linewidth=1)
    # ax.axhline(0.0, color="gray", linestyle=":", linewidth=1)
    # ax.set_yscale("symlog", linthresh=LINTHRESH)
    # ax.yaxis.set_major_locator(
    #     SymmetricalLogLocator(base=10, linthresh=LINTHRESH, subs=[1, 2, 3, 5])
    # )
    # ax.yaxis.set_major_formatter(_ii_fmt)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)

g.set_titles(
    row_template="{row_name}",
    col_template="{col_name}",
    size=FONT_SIZE,
)
middle_col_title = g.axes[0, middle_col].get_title()
g.axes[0, middle_col].set_title(
    f"wielkość zbioru treningowego\n{middle_col_title}", fontsize=FONT_SIZE
)
middle_row_title = g._margin_titles_texts[middle_row]
middle_row_title.set_text(f"współczynnik korelacji\n{middle_row_title.get_text()}")
middle_row_title.set_multialignment("center")
for margin_title in g._margin_titles_texts:
    margin_title.set_fontsize(FONT_SIZE)
if g._legend is not None:
    plt.setp(g._legend.get_texts(), fontsize=FONT_SIZE)
    g._legend.get_title().set_fontsize(FONT_SIZE)

# g.figure.suptitle("ii score vs ranking by cov and training size", y=1.02)
save_figure(g.figure, "recovery_ii_vs_ranking_grid")
plt.show()